# E4 — Tiled inference

E4 dùng `best.pt` của E1, chia ảnh thành tile 640×640, chạy inference từng tile và merge bằng class-aware NMS. Chọn overlap trên validation trước khi chạy test.

In [ ]:
from pathlib import Path
import os, sys, cv2, numpy as np
candidate = Path.cwd().resolve()
for directory in (candidate, *candidate.parents):
    if (directory / 'pyproject.toml').is_file(): PROJECT_ROOT = directory; break
else: raise RuntimeError('Không tìm thấy project root')
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT / 'src'))
E4_CONFIG = PROJECT_ROOT / 'configs/E4_tiled_inference.yaml'
E1_CHECKPOINT = PROJECT_ROOT / 'experiments/E1/baseline_seed42/weights/best.pt'

In [ ]:
from helmet_yolov10.utils.config import load_config
from helmet_yolov10.training.train import _load_backend
from helmet_yolov10.inference.tiled import predict_tiled
config = load_config(E4_CONFIG)
assert E1_CHECKPOINT.is_file(), f'Không tìm thấy: {E1_CHECKPOINT}'
settings = config['tiled_inference']; print(settings)
model = _load_backend()(str(E1_CHECKPOINT))

## Chọn overlap trên validation

Dùng cùng các ảnh validation low-light/small-object đã định nghĩa trước. Ghi accuracy và benchmark cho từng overlap; sau đó đặt `SELECTED_OVERLAP` trước test. Không dùng test để chọn overlap.

In [ ]:
IMAGE = None  # Path tới một ảnh validation hoặc test chỉ để xem prediction
OVERLAP = 0.25
if IMAGE is None:
    print('Đặt IMAGE để chạy demo.')
else:
    image = cv2.imread(str(IMAGE))
    result = predict_tiled(model, image, tile_size=tuple(settings['tile_size']), overlap=OVERLAP, conf=settings['confidence_threshold'], iou=settings['merge_iou_threshold'], device=0)
    print({key: result[key] for key in ('tiles', 'seconds', 'fps')})
    for box, score, cls in zip(result['boxes'].astype(int), result['scores'], result['classes']):
        cv2.rectangle(image, tuple(box[:2]), tuple(box[2:]), (0,255,0), 2)
        cv2.putText(image, f'{cls}: {score:.2f}', tuple(box[:2]), cv2.FONT_HERSHEY_SIMPLEX, .5, (0,255,0), 1)
    import matplotlib.pyplot as plt
    plt.figure(figsize=(12,8)); plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB)); plt.axis('off')

In [ ]:
# Benchmark trên danh sách ảnh cùng test images với standard inference.
IMAGE_PATHS = []  # Điền danh sách Path ảnh.
for overlap in settings['overlap_candidates']:
    runs = [predict_tiled(model, cv2.imread(str(path)), tile_size=tuple(settings['tile_size']), overlap=overlap, conf=settings['confidence_threshold'], iou=settings['merge_iou_threshold'], device=0) for path in IMAGE_PATHS]
    if runs: print({'overlap': overlap, 'time_per_image': np.mean([r['seconds'] for r in runs]), 'fps': np.mean([r['fps'] for r in runs]), 'tiles_per_image': np.mean([r['tiles'] for r in runs])})
SELECTED_OVERLAP = None  # Chỉ đặt sau validation, ví dụ 0.25